In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class BatchNorm(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1, affine=True, track_running_stats=True):
        super().__init__()
        self.num_features = num_features
        self.eps = eps # Avoid 0 division
        self.momentum = momentum # Weight that discards/adds signal to the exp mean and exp variance
        self.affine = affine # If true, has learnable affine parameters
        self.track_running_stats = track_running_stats # If true, tracks the exp mean and exp variance

        if self.affine:
            self.scale = nn.Parameter(torch.ones(num_features)) # Initiate to one for no-op
            self.shift = nn.Parameter(torch.zeros(num_features)) # Initiate to zero for no-op

        if self.track_running_stats:
            self.register_buffer("exp_mean", torch.zeros(num_features)) # Initiate non learnable vector with mean at 0
            self.register_buffer("exp_var", torch.ones(num_features)) # Initiate non learnable vector with variance at 1

    def forward(self, x):
        original_shape = x.shape
        batch_size = x.shape[0]

        x = x.view(batch_size, self.num_features, -1) # Reshape [batch_size, num_features, *] to [batch_size, num_features, n]
        mean = torch.mean(x, dim=(0,2), keepdim=True) # Compute the mean of the batch for each feature
        var = torch.var(x, dim=(0,2), correction=0, keepdim=True) # Compute the var of the batch for each feature, correction=0 for population variance

        if self.training: # self.training is handled by pytorch with model.train() and model.eval()
            norm = (x - mean) / torch.sqrt(var + self.eps) # Compute normalization
            if self.track_running_stats:
                self.exp_mean = (1-self.momentum) * self.exp_mean + self.momentum * mean.view(-1)
                self.exp_var = (1-self.momentum) * self.exp_var + self.momentum * var.view(-1)
        else: # If inference mode
            norm = (x - self.exp_mean.view(1, -1, 1)) / torch.sqrt(self.exp_var.view(1, -1, 1) + self.eps)

        if self.affine:
            norm = self.scale.view(1, -1, 1) * norm + self.shift.view(1, -1, 1) # Scale and shift with learned vectors

        return norm.view(original_shape) # View as original shape

In [ ]:
x = torch.randn(64, 3, 32, 32)
bn = BatchNorm(num_features=3)
bn.train()

out = bn(x) # Forward pass

print(out.mean(dim=(0, 2, 3)))
print(out.std(dim=(0, 2, 3)))

tensor([-2.2701e-09,  3.1432e-09,  1.8277e-08], grad_fn=<MeanBackward1>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<StdBackward0>)
